# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andrew417/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
import duckdb, pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN", None) or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect(database=':memory:')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
REL_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

In [5]:
con.sql(f"SELECT * from {REL_MARCH} LIMIT 5").show()

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [10]:
feat = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {REL_MARCH})
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_prev30,
        SUM(gsc_clicks) AS clicks_prev30,
        AVG(gsc_avg_position) AS avg_position_prev30,
        SUM(ga4_sessions) AS ga4_sessions_prev30,
        SUM(scroll_events) AS scroll_events_prev30
    FROM {REL_MARCH}, bounds
    WHERE client_has_ga4 = true
    GROUP BY content_hash_id, client_hash_id
""").df()
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,impressions_prev30,clicks_prev30,avg_position_prev30,ga4_sessions_prev30,scroll_events_prev30
0,content_7dd54420a11a4cdb,client_3197e6291363b4db,0.0,0.0,NaN,2.0,0.0
1,content_75bc383c7a728be5,client_3197e6291363b4db,0.0,0.0,NaN,0.0,0.0
2,content_8dd083726f6b50dd,client_3197e6291363b4db,0.0,0.0,NaN,1.0,1.0
3,content_b697d1f1ab359d43,client_3197e6291363b4db,0.0,0.0,NaN,2.0,0.0
4,content_565b6d46712e87c0,client_3197e6291363b4db,0.0,0.0,NaN,1.0,1.0


In [11]:
feat.describe()

,impressions_prev30,clicks_prev30,avg_position_prev30,ga4_sessions_prev30,scroll_events_prev30
count,260737.000000,260737.000000,128012.000000,260737.000000,260737.000000
mean,644.383858,1.868189,14.433325,4.985131,0.844157
std,3414.761699,17.281475,15.569405,28.524165,5.468716
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,4.978360,0.000000,0.000000
50%,0.000000,0.000000,8.166667,0.000000,0.000000
75%,141.000000,0.000000,18.584943,1.000000,0.000000
max,617124.000000,5668.000000,297.000000,2730.000000,667.000000


In [28]:
feat['position_tier'] = pd.cut(
    feat['avg_position_prev30'],
    bins=[-2, -1, 3, 10, 20, 50, 1000],
    labels=['no_data', 'top_3', 'page_1', 'striking', 'page_3_5', 'deep']
)

In [26]:
feat['ctr_prev30'] = feat['clicks_prev30'] / feat['impressions_prev30'].replace(0, pd.NA)

In [30]:
feat['avg_position_prev30'] = feat['avg_position_prev30'].fillna(-1)
feat['ctr_prev30'] = feat['ctr_prev30'].fillna(0)
feat.isnull().sum()
feat.head(10)

,content_hash_id,client_hash_id,impressions_prev30,clicks_prev30,avg_position_prev30,ga4_sessions_prev30,scroll_events_prev30,ctr_prev30,position_tier
0,content_7dd54420a11a4cdb,client_3197e6291363b4db,0.0,0.0,-1.0,2.0,0.0,0,no_data
1,content_75bc383c7a728be5,client_3197e6291363b4db,0.0,0.0,-1.0,0.0,0.0,0,no_data
2,content_8dd083726f6b50dd,client_3197e6291363b4db,0.0,0.0,-1.0,1.0,1.0,0,no_data
3,content_b697d1f1ab359d43,client_3197e6291363b4db,0.0,0.0,-1.0,2.0,0.0,0,no_data
4,content_565b6d46712e87c0,client_3197e6291363b4db,0.0,0.0,-1.0,1.0,1.0,0,no_data
5,content_d02d4dabfeabe668,client_3197e6291363b4db,0.0,0.0,-1.0,0.0,0.0,0,no_data
6,content_a77f4edbb8153e9a,client_3197e6291363b4db,0.0,0.0,-1.0,0.0,0.0,0,no_data
7,content_8141f6613eae7c58,client_3197e6291363b4db,0.0,0.0,-1.0,1.0,0.0,0,no_data
8,content_98170395eaa1f9ae,client_3197e6291363b4db,0.0,0.0,-1.0,10.0,1.0,0,no_data
9,content_b32546a3d18ada08,client_3197e6291363b4db,0.0,0.0,-1.0,0.0,0.0,0,no_data


In [31]:
feat['position_tier'].value_counts()

position_tier
no_data     132725
page_1       60764
page_3_5     24062
striking     23935
top_3        13786
deep          5465
Name: count, dtype: int64

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**impressions_prev30** — total GSC impressions summed over March (prev-30 window). Missing values: none after filtering to `client_has_ga4 = true`; zero-impression pages are real zeros, not nulls. Available before decision point: yes — purely historical.

**clicks_prev30** — total GSC clicks summed over March. Same missingness handling as impressions. Available before decision point: yes.

**avg_position_prev30** — mean GSC search position over the window. Missing when a page had zero impressions all month (no position to measure); filled with -1 as a sentinel meaning "no data," matching the dataset's own convention for `avg_position = 0`. Available before decision point: yes.

**ga4_sessions_prev30** — total GA4 sessions summed over March, for rows where `client_has_ga4 = true` only. Rows with `client_has_ga4 = false` are excluded entirely (see section 4), not filled — their GA4 columns are structurally absent, not zero. Available before decision point: yes.

**scroll_events_prev30** — total GA4 scroll events summed over March, same scope and missingness handling as `ga4_sessions_prev30`. Available before decision point: yes.

**ctr_prev30** (engineered) — click-through rate, `clicks_prev30 / impressions_prev30`. Undefined when impressions are 0 (division by zero); filled with 0, meaning "no clicks observed because no visibility to click from." Available before decision point: yes — computed entirely from prev-30 raw features.

**position_tier** (engineered, categorical) — `avg_position_prev30` bucketed into six tiers (no_data, top_3, page_1, striking, page_3_5, deep), following the dataset's own `position_tier` convention. Captures the non-linear relationship between search rank and click behavior (top-3 matters far more than deep positions). Missing values: none — `no_data` bucket explicitly covers the -1 sentinel. Available before decision point: yes — derived only from `avg_position_prev30`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**report_date** — collapsed via aggregation (SUM/AVG over the month), not used as a standalone feature; a raw date isn't meaningful to a model as-is.

**client_has_ga4, ga4_data_available, client_has_gsc, gsc_data_available** — used only as filter/context to scope which rows to include, never as model inputs; they describe data availability, not content performance.

**Rows where client_has_ga4 = false** — excluded entirely (not filled). GA4 columns are structurally NULL for these rows (no tracking access), not real zeros; filling them would misrepresent "never measured" as "zero engagement." Verified in ML-04: 3,018,741 of 9,841,378 March rows (30.7%).

**gsc_sum_position** — redundant with gsc_avg_position (sum vs. mean of the same underlying values); kept the average since it's already normalized for volume.

**ga4_pageviews, ga4_users** — highly correlated with ga4_sessions (all move together with traffic volume); excluded to avoid redundant, near-duplicate signal in a small feature set.

**ga4_engaged_sessions, ga4_total_engagement_sec** — reasonable quality signals but not included in this pass to stay within the assignment's feature budget; worth revisiting as an engagement_rate ratio later.

**sessions_organic/direct/referral/social/paid, ai_chatgpt/perplexity/gemini/copilot/claude/meta/other** — channel-level breakdowns not needed for a content-level refresh score; ga4_sessions already captures total traffic.

**month** — redundant with report_date/the fact this whole slice is already scoped to March; carries no additional information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.